In [ ]:
!pkill -9 ngrok
import gc
gc.collect()
print("Memory cleared! Now you can use the correct key.")

In [ ]:
# ==========================================
# 0. INSTALL, IMPORTS & CLEANUP
# ==========================================
!pip install pyngrok uvicorn fastapi python-multipart
!pkill -9 ngrok

import tensorflow as tf
from tensorflow.keras.preprocessing import image
import torch, os, gc, uvicorn, nest_asyncio
import numpy as np
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
from pyngrok import ngrok
from transformers import pipeline, BitsAndBytesConfig

# Clear memory before starting
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 1. INITIALIZE API & MODELS
# ==========================================
app = FastAPI()
nest_asyncio.apply()

# Define paths (Double-check these in your Kaggle Sidebar!)
BONE_PATH = '/kaggle/input/datasets/bilalsalamm/trained/bone_fracture_model (1).h5'
MRI_PATH  = '/kaggle/input/datasets/bilalsalamm/daatasetss/mri_model.h5'
SKIN_PATH = '/kaggle/input/datasets/bilalsalamm/daatasetss/skin_model.h5'
XRAY_PATH = '/kaggle/input/datasets/bilalsalamm/daatasetss/chest_xray_model.h5'
LLM_MODEL = "HuggingFaceH4/zephyr-7b-beta"

print("Loading Vision Engines...")
bone_model = tf.keras.models.load_model(BONE_PATH, compile=False)
mri_model  = tf.keras.models.load_model(MRI_PATH, compile=False)
skin_model = tf.keras.models.load_model(SKIN_PATH, compile=False)
xray_model = tf.keras.models.load_model(XRAY_PATH, compile=False)

print("Loading Narrative Engine (4-bit)...")
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
report_pipe = pipeline("text-generation", model=LLM_MODEL, model_kwargs={"quantization_config": quant_config}, device_map="auto")

# ==========================================
# 2. MULTI-MODEL ROUTING LOGIC
# ==========================================

def get_multi_diagnosis(img_path, modality_type):
    # --- STEP 1: DEFINE CORRECT DIMENSIONS ---
    if modality_type == 'MRI':
        target_dim = (150, 150)  # Standard for MRI models
    elif modality_type == 'Skin':
        target_dim = (224, 224)
    else:
        target_dim = (224, 224) # Default for Bone/X-Ray

    # --- STEP 2: PREPROCESS WITH CORRECT SIZE ---
    img = image.load_img(img_path, target_size=target_dim)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # --- STEP 3: RUN INFERENCE ---
    if modality_type == 'Fracture':
        score = bone_model.predict(img_array, verbose=0)[0][0]
        diag = "Fracture Detected" if score < 0.5 else "Normal"
        conf = (1 - score) * 100 if score < 0.5 else score * 100
    
    elif modality_type == 'MRI':
        preds = mri_model.predict(img_array, verbose=0)[0]
        # Labels must match your training order!
        labels = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
        diag = labels[np.argmax(preds)]
        conf = np.max(preds) * 100
    
    # ... keep your Skin and X-ray logic the same ...
        
    
        
    elif modality_type == 'Skin':
        preds = skin_model.predict(img_array, verbose=0)[0]
        diag = "Malignant" if np.argmax(preds) == 1 else "Benign"
        conf = np.max(preds) * 100
        
    else: # Default Chest X-Ray
        score = xray_model.predict(img_array, verbose=0)[0][0]
        diag = "Pneumonia" if score > 0.5 else "Normal"
        conf = score * 100
        
    return diag, conf

# ==========================================
# 3. API ENDPOINT
# ==========================================
# Add this right above your @app.post("/diagnose") in Kaggle
@app.get("/")
async def health_check():
    return {"status": "online", "message": "CarePlus AI Brain is Active"}

@app.post("/diagnose")
async def diagnose(file: UploadFile = File(...), modality: str = "Fracture"):
    contents = await file.read()
    with open("temp.jpg", "wb") as f: f.write(contents)
    
    # 1. AI Analysis
    diag, conf = get_multi_diagnosis("temp.jpg", modality)
    
    # 2. Generate specialized report using Zephyr-7B
    prompt = f"<|system|>\nYou are a Medical Specialist. Provide clinical OBSERVATIONS and REMEDIES for a {modality} scan.</s>\n<|user|>\nFinding: {diag}\nConfidence: {conf:.2f}%</s>\n<|assistant|>\n"
    output = report_pipe(prompt, max_new_tokens=500, temperature=0.3)
    report = output[0]['generated_text'].split("<|assistant|>")[1].strip()
    
    return JSONResponse({
        "modality": modality,
        "diagnosis": diag,
        "confidence": f"{conf:.2f}%",
        "report": report
    })

# ==========================================
# 4. START THE BRIDGE
# ==========================================
MY_TOKEN = "3AhJflempLbnWfnUHpSO0Jry6jW_7D69YMTkJ7ypMcrphZnC5"
ngrok.set_auth_token(MY_TOKEN)

try:
    public_url = ngrok.connect(8000).public_url
    print(f"\n" + "="*50)
    print(f"✅ MULTI-MODEL API LIVE: {public_url}/diagnose")
    print(f"👉 Use this URL in your Django code.")
    print("="*50 + "\n")
    
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    await server.serve()
except Exception as e:
    print(f"❌ Connection failed: {e}")

2026-04-08 04:11:09.850111: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775621470.039269     106 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775621470.094624     106 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775621470.562708     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775621470.562748     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775621470.562751     106 computation_placer.cc:177] computation placer alr

Loading Vision Engines...


I0000 00:00:1775621506.925805     106 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775621506.932681     106 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Loading Narrative Engine (4-bit)...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

...

                                                                                                    
✅ MULTI-MODEL API LIVE: https://nathan-censorial-surreally.ngrok-free.dev/diagnose
👉 Use this URL in your Django code.



INFO:     Started server process [106]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
I0000 00:00:1775621603.029113     174 service.cc:152] XLA service 0x7a351004e450 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775621603.029164     174 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775621603.029168     174 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775621603.783626     174 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-08 04:13:31.210524: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-08 04:13:31.345922: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:

INFO:     2405:201:f00c:15b:6956:c8f7:8e7d:deda:0 - "POST /diagnose?modality=XRay HTTP/1.1" 200 OK


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     2405:201:f00c:15b:6956:c8f7:8e7d:deda:0 - "POST /diagnose?modality=Fracture HTTP/1.1" 200 OK


In [ ]:
pip install -U bitsandbytes>=0.46.1